#### 🌟 Why do we use Batch Normalization? 

1. **Reduce Covariate Shift:**  
   Normalize the activations of each layer within every mini-batch, ensuring that the distribution of inputs to layers remains stable throughout training.

2. **Mitigate Vanishing/Exploding Gradients:**  
   Helps address the vanishing or exploding gradient problem, making training deep neural networks more stable.

3. **Faster Convergence:**  
   Speeds up training by allowing for higher learning rates and reducing the number of epochs needed to achieve a desired loss.

---

#### 🚗 Analogy

Imagine learning to drive a car where the sensitivity of the steering wheel changes every day—it would be incredibly hard to learn! Similarly, in deep neural networks (DNNs), if each layer receives inputs of wildly different scales, training becomes challenging and unstable.  
Batch Normalization (BN) solves this by scaling the activations of each mini-batch to have roughly zero mean and unit variance, making learning smoother for every layer.

---

#### 💡 Common Interview Question

**Q:** Why use Batch Normalization? Why not just normalize your input data with a standard scaler?

**A:** Normalizing input data is only helpful for the first layer. In deep neural networks with many layers, the intermediate layers may still experience instability due to changing distributions (i.e., vanishing/exploding gradients). Batch Normalization ensures that all layers receive stable, normalized inputs, promoting better and faster learning throughout the network.


<img src="BN_steps.png" alt="Batch Normalization Steps" width="500"/>

### Implimentation in CNN of BN

#### Where to add BN and why? 
**Q.** **Conv Layer -> Batch Normalization Layer -> Activation Function -> Pooling Layer** 

**A.** If Activation (say ReLU comes before BN) then ReLU would do f(x) = max(0,x), this would lead to loss of information in CNNs, while if BN is before activation then it would normalize the whole i/p distribution of all the mini batch before they enter activation functions. 

**Q. What happens at test time? As you might not always get a batch, sometimes the batch size might be 1, so how will you compute the mean and variance of the batch in that case?**

**A.** During inferencing we do not trust the current batch at inference time, we use what the model learned during training, in short we use learned mean and learned variance from training. So in a nutshell, At test time, BatchNorm uses running mean and running variance instead of new input batch statistics. 

In [2]:
import tensorflow as tf 
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPooling2D

# 
# Conv2D(32, (3,3), ...) means:
#   - 32: number of filters (sometimes called kernels). Higher number allows the network to learn more different types of features.
#   - (3,3): height and width of each filter. (3,3) is common because it's small enough to capture local information but large enough to combine nearby pixels.
# 
# The input_shape = (64, 64, 3):
#   - 64, 64: height and width of the input images.
#   - 3: number of color channels (RGB).
#   - This must be specified only for the very first layer so that Keras knows the shape of the input data.
#   - You might use a different input shape depending on your dataset; for example, (28,28,1) for grayscale MNIST digits, or (224,224,3) for standard color images.

model = tf.keras.Sequential([
    Conv2D(32, (3,3), padding='same', input_shape=(64, 64, 3)),  # 32 filters, each 3x3, with 3-channel (RGB) 64x64 input images
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(64, (3,3), padding='same'),  # Now 64 filters, each 3x3. Input shape is inferred from previous layer.
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D(pool_size=(2,2))
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

d:\Download\Anaconda\envs\road_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Gradient Clipping 

Another popular technique to mitigate the exploding gradient problem, clip gradients during back prop so that they never cross a certain threshold. 

#### Why does it exist? 
In RNNs, BN is tricky to use, so Gradient clippping is normally used. 

In [ ]:
optimizer = keras.optimizers.SGD(clipvalue=1.0)
model.compile(loss = "mse", optimizer = optimizer )


**Gradient clipping** is a technique used during training to prevent **exploding gradients**, majorly used in RNNs as BN is a bit tricky 

It is especially important in:
- **Deep neural networks**
- **Recurrent Neural Networks (RNNs / LSTMs)**
- Any model with long backpropagation paths

---

#### 🔧 How Gradient Clipping Works

Instead of allowing gradients to grow uncontrollably, we **limit (clip)** their values before updating the model weights.

There are two common methods:

---

#### 1️⃣ Clip by Value (**clipvalue**)

Each component of the gradient is clipped to lie within a fixed range.

Example:
- Original gradient: **[0.9, 100]** the vector tends to the right side 
- After **clipvalue = 1.0** → **[0.9, 1]** the vector changes it direction 

⚠️ This can **change the direction** of the gradient vector, which may negatively affect learning.

---

#### 2️⃣ Clip by Norm (**clipnorm**) ✅

If the **overall magnitude (L2 norm)** of the gradient exceeds a threshold, the entire vector is scaled down proportionally.

Example:
- Original gradient: **[0.9, 100]**
- After **clipnorm = 1.0** → **[0.00899964, 0.9999595]**

✅ The **direction is preserved**, only the magnitude is reduced.